# Time-series deep-learning classification (HydraMultiROCKET, LSTM, TSSequencer)

Fold-based cross-validation of three tsai architectures on the raw tri-axial acceleration windows, for every burst length (B1–B4) and correction type (uncorrected, rotdaily, rotbasal). Folds come from `indices_cv_fold_lookup.csv` (`04_build_cv_fold_lookup`): train = `fold != f`, test = `fold == f`, and all sub-windows of a parent burst share a fold.

The seed is reset before each architecture on each fold, so results are reproducible and one architecture's result does not depend on the others. Metrics are computed post hoc with the same sklearn helper as `05a`–`05c` (fastai's `RocAuc` defaults to macro averaging; weighted is used everywhere else).

Each run writes three outputs for the behaviour target (the activity target is also fitted, but its outputs are not saved; see the save cell):
- per-fold metrics (`*_acceleration_dl_metrics.csv`)
- per-fold confusion matrices (`*_acceleration_dl_confusion.csv`)
- per-window out-of-fold predictions with class probabilities (`*_acceleration_dl_oof_predictions.csv`), carrying `Ind_ID`, `burst_id` and `fold`

The class-weight ablation (HR, B1 and B2 uncorrected, behaviour target) is at the end and runs separately, after the main run.

In [ ]:
import gc
import glob
import os
import time

import numpy as np
import pandas as pd
import torch
import tsai.all as ts
from fastai.torch_core import set_seed
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

## Shared helpers
Identical in `05a`, `05b`, `05c` and `05d`, so every model family uses the same metric definitions and output format.

In [ ]:
TEST_ID_COLS = ["Ind_ID", "new_burst", "burst_id", "fold"]
LEAD_COLS = ["burst", "correction_type", "target", "fold", "model"]


def compute_metrics(y_true, y_pred, proba, n_classes, is_binary):
    """Identical metric definitions for every model family.

    Args:
        y_true (np.ndarray): Encoded true labels
        y_pred (np.ndarray): Encoded predicted labels
        proba (np.ndarray): Predicted class probabilities, one column per class
        n_classes (int): Number of classes in the target
        is_binary (bool): Whether this is binary classification

    Returns:
        dict: Global metrics (weighted and macro averages)
    """
    labels = np.arange(n_classes)
    try:
        roc = (
            roc_auc_score(y_true, proba[:, 1])
            if is_binary
            else roc_auc_score(
                y_true, proba, multi_class="ovr", average="weighted", labels=labels
            )
        )
    except ValueError:
        roc = np.nan
    return {
        "roc_auc": roc,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
    }


def confusion_rows(y_true, y_pred, class_names, model_name):
    """Fixed CxC confusion matrix in long format, so folds can be pooled."""
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n))
    return [
        {
            "model": model_name,
            "actual_label": class_names[i],
            "predicted_label": class_names[j],
            "count": int(cm[i, j]),
        }
        for i in range(n)
        for j in range(n)
    ]


def prediction_rows(test_ids, y_true, y_pred, proba, class_names, model_name):
    """One row per test window: IDs, labels and class probabilities."""
    out = test_ids.reset_index(drop=True).copy()
    out.insert(0, "model", model_name)
    names = np.asarray(class_names, dtype=object)
    out["actual_label"] = names[np.asarray(y_true, dtype=int)]
    out["predicted_label"] = names[np.asarray(y_pred, dtype=int)]
    for i, cls in enumerate(class_names):
        out[f"prob_{cls}"] = proba[:, i]
    return out


def finalise(frames, lead_cols):
    """Concatenate per-fold frames and move the identifying columns to the front."""
    frames = [d for d in frames if d is not None and not d.empty]
    if not frames:
        return pd.DataFrame(columns=lead_cols)
    df = pd.concat(frames, ignore_index=True)
    return df[lead_cols + [c for c in df.columns if c not in lead_cols]]


## Data preparation

In [ ]:
def build_ts_arrays(acc, fold_lookup, burst):
    """Build the time-series X array once per file, aligned to fold assignments.

    Built independently of target column and CV fold, so the same X is reused
    across both classification targets and all folds.

    Args:
        acc (pd.DataFrame): Raw accelerometer long-format data straight from
            the parquet file (columns: Ind_ID, new_burst, feature, plus
            numeric time columns). Labels come from fold_lookup, same as the
            fold-based pipelines in notebooks 05a-05c.
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, burst_id, behaviour, activity, fold
        burst (str): Folder-derived burst name (e.g. "Burst_1"); matched
            against fold_lookup's lowercase burst column

    Returns:
        tuple: (X, merged) where X is an (n_samples, 3, n_features) array and
            merged is a DataFrame with columns id, Ind_ID, new_burst, burst_id,
            behaviour, activity, fold, idx -- merged.loc[i, "idx"] indexes X's
            first axis for sample i.
    """
    m = fold_lookup.loc[
        fold_lookup["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"],
    ]

    # Build id directly from Ind_ID/new_burst (the raw parquet has no id column)
    acc = acc.copy()
    acc["new_burst"] = acc["new_burst"].astype(np.int64)
    acc["id"] = acc["Ind_ID"].astype(str) + "_" + acc["new_burst"].astype(str)
    id_mapping = (
        acc[["id", "Ind_ID", "new_burst"]].drop_duplicates().reset_index(drop=True)
    )

    # Inner join drops Remove/unlabelled ids and attaches fold + label values
    merged = id_mapping.merge(m, on=["Ind_ID", "new_burst"], how="inner")
    merged = merged.reset_index(drop=True)
    merged["idx"] = merged.index

    # Identify the feature columns (all numeric columns)
    feature_cols = [col for col in acc.columns if col.isdigit()]
    n_features = len(feature_cols)
    n_samples = len(merged)
    print(f"Number of samples: {n_samples}, number of features: {n_features}")

    X = np.zeros((n_samples, 3, n_features), dtype=np.float64)
    feature_to_idx = {"X": 0, "Y": 1, "Z": 2}

    for i, id_val in enumerate(merged["id"]):
        id_data = acc[acc["id"] == id_val]
        for feature_type, axis_idx in feature_to_idx.items():
            feature_data = id_data[id_data["feature"] == feature_type]
            if len(feature_data) > 0:
                X[i, axis_idx, :] = feature_data[feature_cols].values[0]
            else:
                print(f"Warning: No data found for ID {id_val}, feature {feature_type}")

    return X.astype(np.float64), merged


def create_dsets_for_fold(X, merged, target_col, fold):
    """Create a TSDatasets object for one CV fold from prebuilt X/labels.

    Args:
        X (np.ndarray): Prebuilt (n_samples, 3, n_features) array shared
            across folds/targets (see build_ts_arrays)
        merged (pd.DataFrame): Output of build_ts_arrays
        target_col (str): "activity" or "behaviour"
        fold (int): Held-out fold id; train = fold != this value,
            valid = fold == this value

    Returns:
        TSDatasets: Dataset object ready for use with tsai DataLoaders. The
            valid split is in merged order of the held-out fold's rows.
    """
    train_indices = merged.loc[merged["fold"] != fold, "idx"].tolist()
    valid_indices = merged.loc[merged["fold"] == fold, "idx"].tolist()
    splits = (train_indices, valid_indices)

    y = merged[target_col].to_numpy(dtype="<U20")

    def y_func(labels):
        # If binary task with "Active"/"Inactive", ensure "Active" is encoded as 1
        unique_labels = np.unique(labels)
        if (
            len(unique_labels) == 2
            and "Active" in unique_labels
            and "Inactive" in unique_labels
        ):
            return np.array([0 if label == "Inactive" else 1 for label in labels])
        return labels

    tfms = [None, [ts.Categorize()]]
    dsets = ts.TSDatasets(
        X.astype(np.float64), y=y_func(y), tfms=tfms, splits=splits, inplace=True
    )

    return dsets


## Model functions

In [ ]:
SEED = 42
ARCHS = [
    (ts.HydraMultiRocket, {}),
    (ts.LSTM, {"n_layers": 6, "bidirectional": True}),
    (ts.TSSequencer, {}),
]


def run_models_on_dataset(dset, test_ids, use_class_weight=True, seed=SEED, archs=ARCHS):
    """Train every architecture on one CV fold from an identical RNG state.

    The seed is reset immediately before building the dataloaders and the model
    for EACH architecture, so an architecture's result does not depend on how
    much randomness the previous one consumed.

    Args:
        dset: TSDatasets object for this fold
        test_ids (pd.DataFrame): TEST_ID_COLS for the valid (test) windows, in
            the valid split's order
        use_class_weight (bool): Whether to weight the training loss by
            inverse class frequency via dls.train.cws. This is tsai's
            analogue of class_weight="balanced" in the ML pipeline (same
            inverse-frequency formula), applied as a loss reweighting via
            CrossEntropyLossFlat. cws is computed from this fold's training split.
        seed (int): Seed reset before each architecture
        archs (list): (architecture, params) pairs to train

    Returns:
        tuple: (metrics_df, confusion_df, predictions_df). An architecture
        that fails is skipped (no placeholder row).
    """
    vocab = list(dset.vocab)
    n_classes = len(vocab)
    is_binary = n_classes == 2
    class_names = ["Inactive", "Active"] if is_binary else vocab  # binary vocab is [0, 1]

    metrics, conf, preds = [], [], []
    for arch, params in archs:
        name = arch.__name__
        try:
            set_seed(seed, reproducible=True)
            dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
            loss_func = (
                ts.CrossEntropyLossFlat(weight=dls.train.cws)
                if use_class_weight
                else None
            )
            model = ts.create_model(arch, dls=dls, **params)
            learn = ts.Learner(dls, model, metrics=[], loss_func=loss_func)
            t0 = time.time()
            lr_max = learn.lr_find()
            learn.fit_one_cycle(50, lr_max)
            elapsed = time.time() - t0

            # Valid set, in split order
            proba, y_true, y_pred = learn.get_preds(with_decoded=True)
            proba = proba.cpu().numpy()
            y_true = y_true.cpu().numpy().astype(int)
            y_pred = y_pred.cpu().numpy().astype(int)
            assert len(y_true) == len(test_ids)

            metrics.append(
                {
                    "model": name,
                    "params": str(params),
                    **compute_metrics(y_true, y_pred, proba, n_classes, is_binary),
                    "time": elapsed,
                }
            )
            conf += confusion_rows(y_true, y_pred, class_names, name)
            preds.append(
                prediction_rows(test_ids, y_true, y_pred, proba, class_names, name)
            )
            del learn, model, dls
        except Exception as e:
            print(f"Error with {name}: {e!s}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    preds = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    return pd.DataFrame(metrics), pd.DataFrame(conf), preds

In [ ]:
TARGETS = [
    ("binary", "activity", "Activity"),
    ("multiclass", "behaviour", "Behaviour"),
]


def process_parquet_file(
    file_path,
    fold_lookup,
    use_class_weight=True,
    archs=ARCHS,
    targets=TARGETS,
    max_folds=None,
):
    """Run fold-based CV on one parquet file for each target.

    Args:
        file_path (str): Path to the parquet file with accelerometer data
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, burst_id, behaviour, activity, fold
        use_class_weight (bool): Whether to weight the training loss by
            inverse class frequency via dls.train.cws (see run_models_on_dataset)
        archs (list): (architecture, params) pairs to train
        targets (list): (key, lookup column, target name) triples to run
        max_folds (int, optional): Run only the first max_folds folds (smoke test)

    Returns:
        dict: <key>_metrics, <key>_conf and <key>_pred DataFrames for each target
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    # Load raw accelerometer data and build X once, shared across targets/folds
    acc = pd.read_parquet(file_path)
    X, merged = build_ts_arrays(acc, fold_lookup, burst)
    folds = sorted(merged["fold"].unique())[:max_folds]

    out = {key: {"metrics": [], "conf": [], "pred": []} for key, _, _ in targets}
    for key, col, tgt in targets:
        for f in folds:
            print(f"  Running {tgt} classification, fold {f}")
            dset = create_dsets_for_fold(X, merged, col, f)
            test_ids = merged.loc[merged["fold"] == f, TEST_ID_COLS]
            met, conf, pred = run_models_on_dataset(
                dset, test_ids, use_class_weight=use_class_weight, archs=archs
            )
            for d in (met, conf, pred):
                if not d.empty:
                    d["correction_type"] = correction_type
                    d["burst"] = burst
                    d["target"] = tgt
                    d["fold"] = f
            out[key]["metrics"].append(met)
            out[key]["conf"].append(conf)
            out[key]["pred"].append(pred)

            # Free GPU memory between folds
            del dset
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return {
        f"{key}_{kind}": finalise(out[key][kind], LEAD_COLS)
        for key, _, _ in targets
        for kind in ("metrics", "conf", "pred")
    }

In [ ]:
def run_model_comparisons(
    folder_path,
    fold_id_path="../data/temp/indices_cv_fold_lookup.csv",
    correction_filters=None,
    use_class_weight=True,
):
    """Run fold-based CV model comparisons across all parquet files in a folder.

    Args:
        folder_path (str): Path to folder containing parquet files
        fold_id_path (str): Path to CSV file containing fold IDs
        correction_filters (list, optional): List of correction types to include
        use_class_weight (bool): Whether to weight the training loss by
            inverse class frequency via dls.train.cws

    Returns:
        dict: Dictionary with combined results DataFrames
    """
    # Find all parquet files
    all_files = glob.glob(os.path.join(folder_path, "**/*.parquet"), recursive=True)
    # Filter by correction type
    if correction_filters is not None:
        all_files = [
            f
            for f in all_files
            if any(corr.lower() in f.lower() for corr in correction_filters)
        ]

    print(f"Found {len(all_files)} parquet files")

    fold_lookup = pd.read_csv(fold_id_path)

    # Process all files and combine results
    all_results = [
        process_parquet_file(f, fold_lookup, use_class_weight=use_class_weight)
        for f in all_files
    ]
    keys = [
        f"{t}_{k}"
        for t in ("binary", "multiclass")
        for k in ("metrics", "conf", "pred")
    ]
    return {k: pd.concat([r[k] for r in all_results], ignore_index=True) for k in keys}

## Smoke test (one file, one fold)
Set `RUN_SMOKE_TEST = True` to run it before the overnight run; it is skipped otherwise. Checks that every architecture ran, the OOF output has one row per test window, the probabilities sum to 1 and `burst_id` is filled.

In [ ]:
# Set to True to run the smoke test interactively; False skips it in Run All / nbconvert
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    fold_lookup = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")
    smoke = process_parquet_file(
        "../data/temp/acc_tsai/Burst_2/annotated_acc_tsai_burst_2_uncorrected.parquet",
        fold_lookup,
        max_folds=1,
    )
    for key in ("binary", "multiclass"):
        pred = smoke[f"{key}_pred"]
        prob = pred.filter(like="prob_")
        n_test = (
            (fold_lookup["burst"] == "burst_2") & (fold_lookup["fold"] == pred["fold"].iloc[0])
        ).sum()
        print(
            f"{key}: models={sorted(pred['model'].unique())}, "
            f"rows per model={pred.groupby('model').size().tolist()} (test windows: {n_test}), "
            f"max |prob sum - 1|={(prob.sum(axis=1) - 1).abs().max():.2e}, "
            f"burst_id missing={pred['burst_id'].isna().sum()}"
        )
    display(smoke["multiclass_metrics"])


## Run over all bursts and correction types

In [ ]:
pq_folder_locations = [
    "../data/temp/acc_tsai/Burst_1/",
    "../data/temp/acc_tsai/Burst_2/",
    "../data/temp/acc_tsai/Burst_3/",
    "../data/temp/acc_tsai/Burst_4/",
]
fold_id_path = "../data/temp/indices_cv_fold_lookup.csv"
correction_types_to_process = ["uncorrected", "rotdaily", "rotbasal"]

In [ ]:
all_burst_results = [
    run_model_comparisons(folder, fold_id_path, correction_types_to_process)
    for folder in pq_folder_locations
]

# Combine bursts into single dataframes
res = {
    k: pd.concat([r[k] for r in all_burst_results], ignore_index=True)
    for k in all_burst_results[0]
}

## Check for failed fits (before saving)
Every (dataset, target, model) should have 5 folds; an architecture that fails on a fold leaves no row.

In [ ]:
for k in ("binary_metrics", "multiclass_metrics"):
    n = res[k].groupby(["burst", "correction_type", "model"])["fold"].nunique()
    print(f"{k}: {len(n)} (dataset, model) combinations for this target, {(n < 5).sum()} with missing folds")
    if (n < 5).any():
        print(n[n < 5])


## Save

In [ ]:
# Output to CSV files
out_files = {
    # Activity (active/inactive) outputs are not used in the paper and are not
    # shipped; uncomment to save them (create data/output/comparison_activity/ first)
    # "binary_metrics": "comparison_activity/activity_acceleration_dl_metrics.csv",
    # "binary_conf": "comparison_activity/activity_acceleration_dl_confusion.csv",
    # "binary_pred": "comparison_activity/activity_acceleration_dl_oof_predictions.csv",
    "multiclass_metrics": "comparison_behaviour/behaviour_acceleration_dl_metrics.csv",
    "multiclass_conf": "comparison_behaviour/behaviour_acceleration_dl_confusion.csv",
    "multiclass_pred": "comparison_behaviour/behaviour_acceleration_dl_oof_predictions.csv",
}
for k, path in out_files.items():
    res[k].to_csv(f"../data/output/{path}", index=False)
    print(f"Saved {len(res[k])} rows -> {path}")

## Class-weight ablation (HR, B1 and B2 uncorrected; after the main run)
Re-runs HydraMultiROCKET on B1 uncorrected (the raw-data baseline) and B2 uncorrected (the final classifier's burst length) without class weighting, with the same seed, folds and settings as the main run, so the only difference is the loss weighting. These are the same two datasets as the LOIO validation (`08a`–`08c`). Behaviour target only. Run the imports, helper and function cells first, then these cells; they do not need the main run in memory. See `05a` for the RF/XGB ablation and `05c` for TabPFN.

In [ ]:
ABLATION_BURSTS = ["Burst_1", "Burst_2"]

fold_lookup = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")
nocw_runs = [
    process_parquet_file(
        f"../data/temp/acc_tsai/{b}/annotated_acc_tsai_{b.lower()}_uncorrected.parquet",
        fold_lookup,
        use_class_weight=False,
        archs=[(ts.HydraMultiRocket, {})],
        targets=[("multiclass", "behaviour", "Behaviour")],
    )
    for b in ABLATION_BURSTS
]
nocw = {
    k: pd.concat([r[k] for r in nocw_runs], ignore_index=True)
    for k in nocw_runs[0]
}
print(nocw["multiclass_pred"].groupby("burst").size())

In [ ]:
# Pooled per-behaviour precision/recall/F1, weighted (main run) vs unweighted
def pooled_pr(pred):
    rows = []
    for b in sorted(pred["actual_label"].unique()):
        tp = ((pred["actual_label"] == b) & (pred["predicted_label"] == b)).sum()
        n_true = (pred["actual_label"] == b).sum()
        n_pred = (pred["predicted_label"] == b).sum()
        p = tp / n_pred if n_pred else np.nan
        r = tp / n_true
        rows.append(
            {
                "behaviour": b,
                "n": n_true,
                "precision": p,
                "recall": r,
                "f1": 2 * p * r / (p + r) if (p + r) else 0.0,
                "pred_true": n_pred / n_true,
            }
        )
    return pd.DataFrame(rows).set_index("behaviour")


main = pd.read_csv(
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_oof_predictions.csv"
)
main = main[
    main["burst"].isin(ABLATION_BURSTS)
    & (main["correction_type"] == "uncorrected")
    & (main["model"] == nocw["multiclass_pred"]["model"].iloc[0])
]
unw = nocw["multiclass_pred"]
pd.concat(
    {
        (b, w): pooled_pr(d[d["burst"] == b])
        for b in ABLATION_BURSTS
        for w, d in (("weighted", main), ("unweighted", unw))
    },
    axis=1,
).round(3)

In [ ]:
for kind in ("metrics", "conf", "pred"):
    name = {"metrics": "metrics", "conf": "confusion", "pred": "oof_predictions"}[kind]
    path = f"../data/output/comparison_behaviour/behaviour_acceleration_dl_nocw_{name}.csv"
    nocw[f"multiclass_{kind}"].to_csv(path, index=False)
    print(f"Saved {len(nocw[f'multiclass_{kind}'])} rows -> {path}")